# Multihead Attention

## Why?

The original [transformer architecture](https://arxiv.org/pdf/1706.03762) uses the attention mechanism multiple times in parallel, allowing the model to learn different aspects of how words contribute to meaning in context. 

Each attention head can specialize in capturing distinct linguistic patterns and relationships, producing richer contextual representations than a single attention mechanism alone.

<img src="images/original-transformer-architecture.png" width="20%" />

## Background
In my last [computational essay](causal-attention-essay.ipynb) I discussed causal masked attention mechanism which is single headed.

In order to implement multiheaded attention mechanism, we will be stacking up multiple instances of single headed causal masked attention module.

## Goal

It is still the same as in previous essays, compute the context vector given the input sequence as embeddings.

**Overview**

<img src="images/multihead-attention-overview.png" width="50%" />

**Expanded**

<img src="images/multihead-attention-overview-expanded.png" width="50%" />

## Multihead Attention Implementation v1

Let's start with where we reached in the last computational essay for the single head causal masked attention mechanism.

And then stack them on top of each other inside a new wrapper class `MultiHeadAttention_v1`.

In [1]:
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_window_length, qkv_bias=False, dropout_rate=0.0):
        super().__init__()
        self.d_out = d_out
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.dropout = nn.Dropout(dropout_rate)
        
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_window_length, context_window_length),
           diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)   
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) 

        d_k = keys.shape[-1]
        attn_weights = torch.softmax(
            attn_scores / d_k**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        
        return context_vec

In [2]:
import torch
import torch.nn as nn

class MultiHeadAttention_v1(nn.Module):
    def __init__(self, d_in, d_out, context_window_length, dropout_rate, num_heads, qkv_bias=False):
        super().__init__()
        
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_window_length, dropout_rate, qkv_bias) for _ in range(num_heads)
        ])

    """Returns concatenated context vectors from multiple single attention head."""
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

## Example: Usage of Multihead Attention Implementation v1

In [3]:
torch.manual_seed(123)

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^0)
    [0.55, 0.87, 0.66], # journey  (x^1)
    [0.57, 0.85, 0.64], # starts   (x^2)
    [0.22, 0.58, 0.33], # with     (x^3)
    [0.77, 0.25, 0.10], # one      (x^4)
    [0.05, 0.80, 0.55]  # step     (x^5)
])

batch = torch.stack((inputs, inputs), dim=0)

d_in = 3
d_out = 2
context_window_length = 6

causal_attention_mechanism = MultiHeadAttention_v1(d_in, d_out, context_window_length, dropout_rate=0.0, num_heads=2)
context_vectors = causal_attention_mechanism(batch)

print("\nMultihead Attention Mechanism Output i.e. Context Vectors:\n", context_vectors)


Multihead Attention Mechanism Output i.e. Context Vectors:
 tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)


**Note:** 
- _Each context vector is of shape 2 x 6._
- _As there are two heads in attention mechanism which makes context vectors of each input batch in a concatenated matrix of shape 4 x 6._
- _As there are two batches of inputs then there are total two matrices of 4 x 6 from nMultihead Attention Mechanism._

## Optimized Multihead Attention Implementation

In our `MultiHeadAttention_v1` implementation, the forward pass iterates through all the attention heads sequentially which is not very performant during the training phase and does not make use of performance benefits offered by GPU.

There is an opportunity of parallelism because attention heads are independent of each other and we could leverage GPU accelerator to parallelize the matrices multiplications.

Technically, we can still use the v1 implementation in the actuall LLM training but at the cost of performance limitations.

### Trick
Core trick to improve the performance for multihead attention is to start with a multi-head layer and then internally split this layer into individual attention heads. It is achieved by:

> _"Initializing the bigger matrices for Wq, Wk, Wv instead of initializing smaller dimension weight matrices for each single attention head."_

**Before the optimization trick:**

Following steps are performed for each attention head, which consist of expensive matrix multiplication operation hence makes it slow.

- Compute the `queries = inputs @ Wq`
- Compute the `key = inputs @ Wk`
- Compute the `values = inputs @ Wv`
- Compute the `attention_scores = queries @ keys.T`
- Compute the `attention_weights = softmax(attention_scores)`
- Compute the `context_vectors = attention_weights @ values`

Then at the end combine the all the context vectors output from individual attention head into one giant matrix.

**Before the optimization trick:**

Combine weight matrices Wq, Wk, Wv in one giant matrix which used to be split and separate matrix muliplication operation in individual attention heads. Because matrix multiplication is costly operation.

- Compute the `queries = inputs @ Wq`
- Compute the `keys = inputs @ Wk`
- Compute the `values = inputs @ Wv`

Then (Splitting is cheaper therefore per attention head): 

- Split the giant `queries` into smaller query matrix per attention head.
- Split the giant `keys` into smaller query matrix per attention head.
- Split the giant `values` into smaller query matrix per attention head.

> _Consolidating weight matrices multiplications and then splitting it later into multiple attention heads is equivalent operation in linear algebra sense._

Followed by:

- Compute the `attention_scores = queries @ keys.T`
- Compute the `attention_weights = softmax(attention_scores)`
- Compute the `context_vectors = attention_weights @ values`

Then at the end combine the all the context vectors output from individual attention heads into one giant matrix.


**Note:** _For this trick to work: we must be having `d_out` divisible by `num_of_heads` means number of output dimensions of projected Q,K,V vectors is a multiple number of attention heads._

<img src="images/multihead-attention-split-weights.png" width="50%" />




In [4]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_window_length, dropout_rate, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout_rate = nn.Dropout(dropout_rate)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_window_length, context_window_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)  
        queries = queries.view(                                             
            b, num_tokens, self.num_heads, self.head_dim                    
        )                                                                   

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        
        attn_weights = self.dropout(attn_weights)

        context_vectors = (attn_weights @ values).transpose(1, 2)

        context_vectors = context_vectors.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vectors = self.out_proj(context_vectors)
        
        return context_vectors

## What did we learn?
- Why original transformer architecture uses multiple attention heads instead of single one to capture distinct linguistic patterns in each attention head.
- How to stack various single attention heads to bring about multiattention head naive implementation v1.
- Why multiattention v1 implementation is slow due to sequention for loop execution.
- How we can optimize multiattention head implementation by leveraging parallelism available in matrix multiplication.